# 02 — Ingestão de Dados Macroeconômicos

## Objetivo

Baixar todas as séries macroeconômicas relevantes para o estudo a partir do Sistema 
Gerenciador de Séries Temporais (SGS) do Banco Central do Brasil e do boletim Focus, 
consolidando tudo em um único CSV mensal em `data/raw/`.

## Variáveis selecionadas

| Variável | Código SGS | Justificativa | Início |
|----------|-----------|---------------|--------|
| `ibcbr` | 24364 | Proxy mensal do PIB (dessazonalizado pelo BCB) | 2003-01 |
| `ipca` | 433 | Inflação ao consumidor (índice oficial de metas) | 2000-01 |
| `selic` | 4189 | Taxa básica de juros (Selic over anualizada, mensal) | 2000-01 |
| `cambio` | 3698 | Câmbio R$/US$ (média mensal, venda) | 2000-01 |
| `commodities` | 27574 | IC-Br: índice de commodities BCB em BRL | 2000-01 |
| `m1` | 27791 | Agregado monetário M1 (saldos em final de período) | 2002-01 |
| `prod_industrial` | 21859 | Produção industrial geral | 2002-01 |
| `credito_total` | 20539 | Saldo total das operações de crédito do SFN | 2000-01 |
| `exp_ipca_12m` | (Focus) | Mediana das expectativas de IPCA 12 meses à frente | 2001-12 |

## Decisões metodológicas

**Selic over (4189) em vez de Meta Selic (432):** a série 432 é diária, sujeita ao limite 
de 10 anos por consulta imposto pelo BCB. A série 4189 é mensal nativa, com diferença 
média irrelevante (~0,1 p.p.) em relação à meta.

**Câmbio em média mensal (3698) em vez de fim de período:** média mensal é mais robusta 
a flutuações pontuais de fim de mês. Para análise macro de médio prazo, é a escolha padrão.

**Focus suavizado, mediana, baseCalculo=0:** série mais estável e abrangente das expectativas. 
A versão "crua" (não suavizada) é mais ruidosa e a baseCalculo 1 e 2 restringem a período 
recente.

**Período de download:** 2000-01-01 a 2025-12-31. O modelo final usa janela menor 
(2003-01 a 2025-12, limitada pelo início do IBC-Br), mas baixamos toda a história 
disponível para permitir análises de robustez em janelas alternativas.

## Outputs

`data/raw/series_macroeconomicas.csv` — DataFrame mensal com índice `DatetimeIndex` 
no primeiro dia de cada mês, 312 observações por série (com NaNs para os períodos 
anteriores ao início de cada uma).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bcb import sgs, Expectativas
from pathlib import Path

# Configurações de visualização
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
plt.rcParams['figure.figsize'] = (12, 5)

# Caminho da pasta de dados (Path lida com / e \ automaticamente entre OS)
DATA_RAW = Path('../data/raw')

In [12]:
# Dicionário com nome amigável como chave e código SGS como valor
series_sgs = {
    'ibcbr':           24364,  # IBC-Br dessazonalizado (mensal, começa 2003-01)
    'ipca':            433,    # IPCA variação mensal (mensal)
    'selic':           4189,   # Selic over acumulada no mês anualizada (mensal)
    'cambio':          3698,   # Câmbio R$/US$ venda, média mensal (mensal)
    'commodities':     27574,  # Índice de Commodities BCB - IC-Br (mensal)
    'm1':              27791,  # M1 Saldos em final de período (mensal)
    'prod_industrial': 21859,  # Produção industrial geral (mensal)
    'credito_total':   20539,  # Saldo das operações de crédito - total (mensal)
}

print(f"Total de séries a baixar: {len(series_sgs)}")

Total de séries a baixar: 8


In [13]:
# Período: jan/2000 até dez/2025
df_sgs = sgs.get(series_sgs, start='2000-01-01', end='2025-12-31')
print(f"Shape: {df_sgs.shape}")
print(f"Período: {df_sgs.index.min()} a {df_sgs.index.max()}")
df_sgs.head()

Shape: (312, 8)
Período: 2000-01-01 00:00:00 a 2025-12-01 00:00:00


,ibcbr,ipca,selic,cambio,commodities,m1,prod_industrial,credito_total
Date,,,,,,,,
2000-01-01,NaN,0.62,18.94,1.8037,51.41,NaN,NaN,289197
2000-02-01,NaN,0.13,18.87,1.7753,50.22,NaN,NaN,287573
2000-03-01,NaN,0.22,18.85,1.7420,49.51,NaN,NaN,286084
2000-04-01,NaN,0.42,18.62,1.7682,50.31,NaN,NaN,291609
2000-05-01,NaN,0.01,18.51,1.8279,53.18,NaN,NaN,312232


In [14]:
# Quantas observações por coluna (NaN = vazio)
print("Observações não-nulas por série:")
print(df_sgs.notna().sum())

print("\nPrimeira data disponível por série:")
print(df_sgs.apply(lambda col: col.first_valid_index()))

print("\nÚltima data disponível por série:")
print(df_sgs.apply(lambda col: col.last_valid_index()))

Observações não-nulas por série:
ibcbr              276
ipca               312
selic              312
cambio             312
commodities        312
m1                 288
prod_industrial    288
credito_total      312
dtype: int64

Primeira data disponível por série:
ibcbr             2003-01-01
ipca              2000-01-01
selic             2000-01-01
cambio            2000-01-01
commodities       2000-01-01
m1                2002-01-01
prod_industrial   2002-01-01
credito_total     2000-01-01
dtype: datetime64[us]

Última data disponível por série:
ibcbr             2025-12-01
ipca              2025-12-01
selic             2025-12-01
cambio            2025-12-01
commodities       2025-12-01
m1                2025-12-01
prod_industrial   2025-12-01
credito_total     2025-12-01
dtype: datetime64[us]


In [16]:
# Como tudo já é mensal, só precisamos garantir índice alinhado ao início do mês
df_sgs_mensal = df_sgs.resample('MS').last()

print(df_sgs_mensal.head())
print(f"\nShape final: {df_sgs_mensal.shape}")

            ibcbr  ipca  selic  cambio  commodities  m1  prod_industrial  credito_total
Date                                                                                   
2000-01-01    NaN  0.62  18.94  1.8037        51.41 NaN              NaN         289197
2000-02-01    NaN  0.13  18.87  1.7753        50.22 NaN              NaN         287573
2000-03-01    NaN  0.22  18.85  1.7420        49.51 NaN              NaN         286084
2000-04-01    NaN  0.42  18.62  1.7682        50.31 NaN              NaN         291609
2000-05-01    NaN  0.01  18.51  1.8279        53.18 NaN              NaN         312232

Shape final: (312, 8)


## Boletim Focus — expectativa de inflação 12 meses à frente

A API do Focus segue padrão OData (similar a `dplyr` em R), diferente do SGS clássico. 
Inclusão dessa série é a principal novidade metodológica em relação à monografia. A 
expectativa de inflação é o principal candidato a resolver o *price puzzle* identificado 
no trabalho original: quando o modelo VAR não inclui informação sobre expectativas, o 
choque positivo na Selic tende a vir acompanhado de subida (em vez de queda) inicial 
nos preços, pois o BCB já estava reagindo a expectativas inflacionárias que o modelo 
não observa diretamente.

In [17]:
# Expectativas de inflação 12 meses à frente
em = Expectativas()
ep = em.get_endpoint('ExpectativasMercadoInflacao12Meses')

# Filtra IPCA e ordena
df_focus = (
    ep.query()
      .filter(ep.Indicador == 'IPCA', ep.Suavizada == 'S', ep.baseCalculo == 0)
      .select(ep.Data, ep.Mediana)
      .collect()
)

# Renomeia colunas e converte data
df_focus.columns = ['data', 'exp_ipca_12m']
df_focus['data'] = pd.to_datetime(df_focus['data'])
df_focus = df_focus.set_index('data').sort_index()

print(df_focus.head())
print(f"\nShape: {df_focus.shape}")
print(f"Período: {df_focus.index.min()} a {df_focus.index.max()}")

            exp_ipca_12m
data                    
2001-12-12          5.22
2001-12-13          5.22
2001-12-14          5.15
2001-12-17          5.16
2001-12-18          5.13

Shape: (6124, 1)
Período: 2001-12-12 00:00:00 a 2026-05-08 00:00:00


In [18]:
# Focus é diário, consolidamos para mensal usando média
df_focus_mensal = df_focus['exp_ipca_12m'].resample('MS').mean().to_frame()

# Junta com o resto
df_completo = df_sgs_mensal.join(df_focus_mensal, how='left')

print(df_completo.head())
print(f"\nShape final: {df_completo.shape}")
print(f"\nObservações não-nulas por série:")
print(df_completo.notna().sum())

            ibcbr  ipca  selic  cambio  commodities  m1  prod_industrial  credito_total  exp_ipca_12m
Date                                                                                                 
2000-01-01    NaN  0.62  18.94  1.8037        51.41 NaN              NaN         289197           NaN
2000-02-01    NaN  0.13  18.87  1.7753        50.22 NaN              NaN         287573           NaN
2000-03-01    NaN  0.22  18.85  1.7420        49.51 NaN              NaN         286084           NaN
2000-04-01    NaN  0.42  18.62  1.7682        50.31 NaN              NaN         291609           NaN
2000-05-01    NaN  0.01  18.51  1.8279        53.18 NaN              NaN         312232           NaN

Shape final: (312, 9)

Observações não-nulas por série:
ibcbr              276
ipca               312
selic              312
cambio             312
commodities        312
m1                 288
prod_industrial    288
credito_total      312
exp_ipca_12m       289
dtype: int64


## Persistência

Salvamos o DataFrame consolidado em CSV para que os notebooks seguintes não precisem 
re-baixar os dados a cada execução. O arquivo fica em `data/raw/`, que está no 
`.gitignore` — reprodutibilidade vem do código, não dos dados versionados.

Em projetos profissionais com dados que podem ser revisados pela fonte, vale considerar 
versionar uma "foto" dos dados em uma data específica (DVC, Git LFS), mas para este 
trabalho a estratégia "re-baixa quando precisa" é suficiente.

In [19]:
# Salva o DataFrame consolidado para não precisar baixar de novo toda vez
caminho_saida = DATA_RAW / 'series_macroeconomicas.csv'
df_completo.to_csv(caminho_saida, index=True, date_format='%Y-%m-%d')

print(f"Arquivo salvo em: {caminho_saida.absolute()}")
print(f"Tamanho: {caminho_saida.stat().st_size / 1024:.1f} KB")

Arquivo salvo em: c:\Users\lucas\Documents\politica-monetaria-br\notebooks\..\data\raw\series_macroeconomicas.csv
Tamanho: 25.4 KB


In [20]:
# Lê de volta para confirmar que tudo está ok
df_verificacao = pd.read_csv(
    caminho_saida,
    index_col=0,
    parse_dates=True
)

print(df_verificacao.head())
print(f"\nÍndice é DatetimeIndex? {isinstance(df_verificacao.index, pd.DatetimeIndex)}")
print(f"\nResumo estatístico:")
print(df_verificacao.describe())

            ibcbr  ipca  selic  cambio  commodities  m1  prod_industrial  credito_total  exp_ipca_12m
Date                                                                                                 
2000-01-01    NaN  0.62  18.94  1.8037        51.41 NaN              NaN         289197           NaN
2000-02-01    NaN  0.13  18.87  1.7753        50.22 NaN              NaN         287573           NaN
2000-03-01    NaN  0.22  18.85  1.7420        49.51 NaN              NaN         286084           NaN
2000-04-01    NaN  0.42  18.62  1.7682        50.31 NaN              NaN         291609           NaN
2000-05-01    NaN  0.01  18.51  1.8279        53.18 NaN              NaN         312232           NaN

Índice é DatetimeIndex? True

Resumo estatístico:
            ibcbr        ipca       selic      cambio  commodities            m1  prod_industrial  credito_total  exp_ipca_12m
count  276.000000  312.000000  312.000000  312.000000   312.000000  2.880000e+02       288.000000   3.120000